# ELF Prompt 0 — Evaluation harness, baseline reproduction & gates

Self-contained notebook for **Prompt 0** of `RESEARCH_ROADMAP.md`. Run it top to bottom.

**Before you start**
1. Pick an **A100** runtime (*Runtime → Change runtime type*). An L4 works for the smoke test only.
2. Upload `elf_code.zip` to `MyDrive/elf_research/code/elf_code.zip`. If it's missing, the setup cell clones the ELF fork from GitHub instead.
3. Optional: add a Colab secret named `HF_TOKEN` (key icon on the left) for faster Hugging Face downloads.

**What it does**
| Section | Purpose |
|---|---|
| 0 Config / Setup | GPU check, Drive, code unzip, pip installs (Colab's preinstalled `transformers` is kept; **don't pin `<4.45`**, because building tokenizers fails on Python 3.13) |
| 1 `elfx/` package | Written to Drive with `%%writefile` so later notebooks can `import elfx` |
| 2 Smoke test | S1–S6 plus an equivalence test against the official sampler. **Everything stops if any check fails.** |
| 3 Full run | R0 reference anchor, R1 ELF-B baselines (3 seeds), official-path cross-check, R2 ELF-M/L, R3 XSum/WMT |
| 4 Gates + table | `runs/_gate.json`, `runs/_baselines_table.md` |

Every run is **idempotent**. If Colab disconnects, re-run all cells and finished configs are skipped.

**Bugs fixed relative to the previous Colab-agent notebook (`elf.ipynb`):**
- **Sampler crash:** the old sampler called `_forward_sample_self_cond` with `cond_seq=None`. The new sampler uses `_forward_sample` with zero cond tensors, exactly as the official code does.
- **Decoding the wrong tensor:** it decoded `x_pred` instead of the final latent `z`.
- **Irreproducible seeds:** it seeded with `hash()`, which is salted per process; seeds now come from sha256.
- **Silent partial load:** it loaded weights with `strict=False` and never checked keys. Keys are now asserted.
- **Stub metrics:** metrics were placeholders. They now reuse the official `Metrics` internals.

In [ ]:
# === 0a. CONFIG — edit here ===
ROOT = '/content/drive/MyDrive/elf_research'   # keep in sync with the %%writefile paths below
RUN_SMOKE = True                 # mandatory before the full run
RUN_FULL = True                  # R0-R3 (~40 CCU on A100)
USE_COMPILE = True               # torch.compile with automatic eager fallback
N_SAMPLES = 1000                 # per unconditional config
SEEDS = [0, 1, 2]
RUN_OFFICIAL_CROSSCHECK = True   # generation.test_generation_uncond end-to-end (1 seed)
RUN_ELF_M = True
RUN_ELF_L = True                 # paper: 23.3 / 5.28
RUN_COND = True                  # XSum + WMT14 de-en via the official path
CCU_PER_HOUR = {'H100': 20.0, 'A100': 12.0, 'L4': 5.0, 'T4': 2.0}   # check Colab's Resources panel
ELF_GIT_FALLBACK = 'https://anonymous.4open.science/r/ELF'      # used only if the zip is missing
ELF_GIT_BRANCH = 'pytorch_elf'

In [ ]:
# === 0b. SETUP ===
import os, sys, json, subprocess, shutil, time
import torch

assert torch.cuda.is_available(), ('No GPU attached. Runtime -> Change runtime type -> A100 GPU, '
                                   'then Runtime -> Restart session and run again.')

from google.colab import drive
drive.mount('/content/drive')

CODE, RUNS, REFS = f'{ROOT}/code', f'{ROOT}/runs', f'{ROOT}/refs'
for d in [ROOT, CODE, RUNS, REFS, f'{CODE}/elfx']:
    os.makedirs(d, exist_ok=True)

# --- ELF source -> /content/repo/ELF ---
zip_path = f'{CODE}/elf_code.zip'
if not os.path.isdir('/content/repo/ELF/src'):
    if os.path.exists(zip_path):
        subprocess.run(['unzip', '-q', '-o', zip_path, '-d', '/content/repo'], check=True)
    else:
        print(f'[setup] {zip_path} not found -> cloning {ELF_GIT_FALLBACK}@{ELF_GIT_BRANCH}')
        os.makedirs('/content/repo', exist_ok=True)
        subprocess.run(['git', 'clone', '-q', '--depth', '1', '-b', ELF_GIT_BRANCH, ELF_GIT_FALLBACK,
                        '/content/repo/ELF'], check=True)
assert os.path.isfile('/content/repo/ELF/src/generation.py'), 'ELF source missing (bad zip layout?)'
os.environ['ELF_ROOT'] = ROOT
os.environ['ELF_SRC'] = '/content/repo/ELF/src'
for p in ['/content/repo/ELF/src', CODE]:
    if p not in sys.path:
        sys.path.insert(0, p)

# --- deps: keep Colab's torch/transformers/numpy; add only what's missing ---
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sacrebleu', 'rouge-score', 'mauve-text',
                'einops', 'sentencepiece'], check=False)

# --- optional HF token ---
try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN')
    if tok:
        os.environ['HF_TOKEN'] = tok
        print('[setup] HF_TOKEN loaded from Colab secrets')
except Exception:
    pass

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')

import transformers, datasets, numpy as np
env = dict(gpu=torch.cuda.get_device_name(0),
           vram_gb=round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1),
           torch=torch.__version__, transformers=transformers.__version__, datasets=datasets.__version__,
           numpy=np.__version__, python=sys.version.split()[0], cuda=torch.version.cuda)
json.dump(env, open(f'{RUNS}/_env.json', 'w'), indent=2)
print(json.dumps(env, indent=2))
if 'A100' not in env['gpu'] and 'H100' not in env['gpu']:
    print('[setup][WARN] Not an A100/H100: fine for the smoke test, slow for the full run.')

## 1. The `elfx/` package (written to Drive)
Each cell below writes one module to `MyDrive/elf_research/code/elfx/`. Later notebooks (1A/1B/2A/2B/4) only need `sys.path.insert(0, CODE); import elfx`.

In [ ]:
%%writefile /content/drive/MyDrive/elf_research/code/elfx/__init__.py
"""elfx: reusable evaluation harness for Embedded Language Flows (ELF, PyTorch port).

Paths are configurable through environment variables so the same code runs in Colab
(defaults below) and locally (set ELF_ROOT / ELF_SRC before importing).
"""
import os
import sys

ROOT = os.environ.get("ELF_ROOT", "/content/drive/MyDrive/elf_research")
ELF_SRC = os.environ.get("ELF_SRC", "/content/repo/ELF/src")
ELF_REPO = os.path.dirname(ELF_SRC.rstrip("/"))
CODE = os.path.join(ROOT, "code")
RUNS = os.path.join(ROOT, "runs")
REFS = os.path.join(ROOT, "refs")

__version__ = "0.2.0"


def _patch_t5_3d_mask():
    """Make 3D (B, L, L) 0/1 attention masks work with transformers >= 4.53 / 5.x T5EncoderModel."""
    try:
        import torch
        from transformers import masking_utils
        orig = getattr(masking_utils, "_preprocess_mask_arguments", None)
        if orig is not None and not getattr(orig, "_elfx_3d_patched", False):
            def _patched_preprocess(config, inputs_embeds, attention_mask, *args, **kwargs):
                if isinstance(attention_mask, torch.Tensor) and attention_mask.ndim == 3:
                    dtype = inputs_embeds.dtype
                    min_val = torch.finfo(torch.bfloat16).min if dtype in (torch.float32, torch.bfloat16) else torch.finfo(dtype).min
                    mask_bool = attention_mask[:, None, :, :].to(device=inputs_embeds.device, dtype=torch.bool)
                    attention_mask = torch.where(mask_bool, torch.zeros((), dtype=dtype, device=inputs_embeds.device), min_val)
                return orig(config, inputs_embeds, attention_mask, *args, **kwargs)
            _patched_preprocess._elfx_3d_patched = True
            masking_utils._preprocess_mask_arguments = _patched_preprocess
    except Exception:
        pass


def ensure_elf_on_path():
    """Put ELF/src first on sys.path (same as ELF/scripts/launch.sh PYTHONPATH)."""
    if ELF_SRC not in sys.path:
        sys.path.insert(0, ELF_SRC)
    _patch_t5_3d_mask()


ensure_elf_on_path()
for _d in (ROOT, RUNS, REFS):
    os.makedirs(_d, exist_ok=True)

In [ ]:
%%writefile /content/drive/MyDrive/elf_research/code/elfx/load.py
"""Model loading that mirrors ELF/src/eval.py (lines 118-152), with strict key checks.

Differences from eval.py (all deliberate):
  * The optimizer is NOT rebuilt (eval.py builds Muon only to satisfy load_checkpoint);
    we read the checkpoint ourselves and load params/EMA with strict key assertions.
  * vocab_size is taken from tokenizer.vocab_size (as eval.py does) but asserted against
    the checkpoint's unembedding shape.
"""
import contextlib
import os
import re
import time

import torch

from . import ELF_REPO, ensure_elf_on_path

ensure_elf_on_path()

# name -> (HF repo, training YAML, expected #params of the eval model)
MODELS = {
    "ELF-B": ("embedded-language-flows/ELF-B-owt-torch", "train_owt_ELF-B.yml", 105e6),
    "ELF-M": ("embedded-language-flows/ELF-M-owt-torch", "train_owt_ELF-M.yml", 342e6),
    "ELF-L": ("embedded-language-flows/ELF-L-owt-torch", "train_owt_ELF-L.yml", 652e6),
    "ELF-B-xsum": ("embedded-language-flows/ELF-B-xsum-torch", "train_xsum_ELF-B.yml", 105e6),
    "ELF-B-de-en": ("embedded-language-flows/ELF-B-de-en-torch", "train_de-en_ELF-B.yml", 105e6),
}
PARAM_TOL = 0.05


@contextlib.contextmanager
def _cwd(path):
    old = os.getcwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(old)


def load_config(name_or_yaml):
    """Load an ELF Config. Relative sampling_configs_path values resolve from the ELF repo root."""
    from configs.config import load_config_from_yaml

    yml = MODELS[name_or_yaml][1] if name_or_yaml in MODELS else name_or_yaml
    path = yml if os.path.isabs(yml) else os.path.join(ELF_REPO, "src", "configs", "training_configs", yml)
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Config YAML not found: {path}")
    with _cwd(ELF_REPO):
        config = load_config_from_yaml(path)
    # Never talk to wandb / HF upload from the harness.
    config.use_wandb = False
    config.hf_repo_id = None
    return config


def _checkpoint_file(repo_or_path):
    """Return a local path to the latest `checkpoint_<step>` of a local dir/file or HF repo."""
    if os.path.isfile(repo_or_path):
        return repo_or_path
    pat = re.compile(r"checkpoint_(\d+)$")
    if os.path.isdir(repo_or_path):
        c = [f for f in os.listdir(repo_or_path) if pat.search(f)]
        if not c:
            raise FileNotFoundError(f"No checkpoint_<step> in {repo_or_path}")
        c.sort(key=lambda f: int(pat.search(f).group(1)))
        return os.path.join(repo_or_path, c[-1])
    from huggingface_hub import hf_hub_download, list_repo_files

    files = [f for f in list_repo_files(repo_or_path) if pat.search(f.split("/")[-1])]
    if not files:
        raise FileNotFoundError(f"No checkpoint_<step> file in HF repo {repo_or_path}")
    files.sort(key=lambda f: int(pat.search(f).group(1)))
    return hf_hub_download(repo_or_path, files[-1])


def _torch_load(path):
    kw = dict(map_location="cpu", weights_only=False)  # official checkpoint; contains optimizer state
    try:
        return torch.load(path, mmap=True, **kw)
    except Exception:
        return torch.load(path, **kw)


def load_elf(name, device=None, return_state=False, checkpoint=None, verbose=True):
    """Load an ELF model exactly like eval.py and return the EMA eval model.

    Returns (model, encoder, tokenizer, config) or, with return_state=True,
    (model, encoder, tokenizer, config, state) where `state` is a utils.train_utils.TrainState
    usable by the official generation.test_generation_{uncond,cond} functions.
    model._elfx_info holds load diagnostics.
    """
    from transformers import AutoTokenizer
    from generation import _build_eval_model
    from modules.model import ELF_models
    from modules.t5_encoder import get_encoder
    from utils.train_utils import TrainState

    if name not in MODELS:
        raise ValueError(f"{name!r} not in {sorted(MODELS)}")
    device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    repo, _, expected = MODELS[name]
    t0 = time.time()

    config = load_config(name)
    tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name or config.encoder_model_name)

    # Encoder (HuggingFace T5), fp32 like eval.py
    encoder_config, encoder = get_encoder(config.encoder_model_name, torch.float32)
    encoder = encoder.to(device).eval()
    for p in encoder.parameters():
        p.requires_grad_(False)

    ckpt_path = _checkpoint_file(checkpoint or repo)
    ckpt = _torch_load(ckpt_path)
    for k in ("params", "step", "epoch"):
        if k not in ckpt:
            raise KeyError(f"checkpoint {ckpt_path} missing key {k!r}; keys={list(ckpt)}")
    params = ckpt["params"]
    has_ema = bool(ckpt.get("ema_params1"))
    ema_sd = ckpt["ema_params1"] if has_ema else params

    vocab_size = tokenizer.vocab_size  # eval.py: vocab_size = tokenizer.vocab_size
    ckpt_vocab = int(params["unembed_bias"].shape[0])
    if vocab_size != ckpt_vocab:
        print(f"[load_elf][WARN] tokenizer.vocab_size={vocab_size} != checkpoint vocab {ckpt_vocab}; "
              f"using {ckpt_vocab} (transformers {__import__('transformers').__version__}).")
        vocab_size = ckpt_vocab

    model = ELF_models[config.model](
        text_encoder_dim=encoder_config.d_model, max_length=config.max_length,
        attn_drop=config.attn_dropout, proj_drop=config.proj_dropout,
        num_time_tokens=config.num_time_tokens,
        num_self_cond_cfg_tokens=config.num_self_cond_cfg_tokens,
        vocab_size=vocab_size,
        num_model_mode_tokens=config.num_model_mode_tokens,
        bottleneck_dim=config.bottleneck_dim,
    )
    res = model.load_state_dict(params, strict=False)
    if res.missing_keys or res.unexpected_keys:
        raise RuntimeError(f"[load_elf] state_dict mismatch for {name}: "
                           f"missing={res.missing_keys[:10]} unexpected={res.unexpected_keys[:10]}")
    model = model.to(device).eval()

    pnames = {n for n, _ in model.named_parameters()}
    ema_missing, ema_unexpected = pnames - set(ema_sd), set(ema_sd) - pnames
    if ema_missing or ema_unexpected:
        raise RuntimeError(f"[load_elf] EMA mismatch: missing={sorted(ema_missing)[:10]} "
                           f"unexpected={sorted(ema_unexpected)[:10]}")
    dev_map = {n: p.device for n, p in model.named_parameters()}
    ema_dev = {n: t.to(dev_map[n]) for n, t in ema_sd.items()}

    state = TrainState(model=model, optimizer=None, lr_scheduler=None,
                       ema_params1=ema_dev, step=int(ckpt["step"]), epoch=int(ckpt["epoch"]))
    del ckpt, params, ema_sd

    eval_model = _build_eval_model(state, use_compile=False)  # deep copy + strict EMA load + eval()
    for p in eval_model.parameters():
        p.requires_grad_(False)

    # Verify the EMA weights (not the raw weights) are in the eval model.
    probe = "final_layer.linear.weight" if "final_layer.linear.weight" in ema_dev else sorted(ema_dev)[0]
    ema_ok = torch.equal(dict(eval_model.named_parameters())[probe], ema_dev[probe])
    n_params = sum(p.numel() for p in eval_model.parameters())
    rel = abs(n_params - expected) / expected
    if rel > PARAM_TOL:
        raise AssertionError(f"[load_elf] {name}: {n_params/1e6:.1f}M params vs expected ~{expected/1e6:.0f}M")

    info = dict(name=name, repo=repo, checkpoint=os.path.basename(ckpt_path), step=state.step,
                epoch=state.epoch, has_ema=has_ema, ema_loaded=bool(ema_ok and has_ema),
                n_params=n_params, vocab_size=vocab_size, missing_keys=[], unexpected_keys=[],
                device=str(device), load_s=round(time.time() - t0, 1))
    eval_model._elfx_info = info
    if verbose:
        print(f"[load_elf] {name}: {n_params/1e6:.1f}M params | step {state.step} | EMA loaded: "
              f"{info['ema_loaded']} | missing/unexpected keys: 0/0 | vocab {vocab_size} | {info['load_s']}s")

    if return_state:
        return eval_model, encoder, tokenizer, config, state
    # Free the raw (non-EMA) training weights; only the EMA eval copy is needed.
    state.model = None
    state.ema_params1 = {}
    del state, model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return eval_model, encoder, tokenizer, config

In [ ]:
%%writefile /content/drive/MyDrive/elf_research/code/elfx/sample.py
"""Sampling for ELF with per-sample reproducible randomness and an optional per-step hook.

Math is identical to ELF/src/utils/generation_utils._generate_samples_single_batch and
utils/sampling_utils._{ode,sde}_step (the network call itself IS the official
_forward_sample), with two generalizations:
  1. t can differ per sample (each sample draws its own logit-normal schedule), which makes
     results independent of batch size.
  2. all randomness (initial noise, schedule, SDE noise) is drawn per sample index from
     generators seeded by (seed, index), using sha256 (Python's hash() is salted per process).

The last step is always an ODE step (as in the official sampler). Decoding uses the official
_dlm_decode_batch on the final latent z, then mask_after_eos, then tokenizer.decode.
"""
import hashlib
import time

import numpy as np
import torch
import torch.nn as nn

from . import ensure_elf_on_path

ensure_elf_on_path()
from utils.data_utils import get_pad_token_id  # noqa: E402
from utils.generation_utils import _dlm_decode_batch, mask_after_eos  # noqa: E402
from utils.sampling_utils import _forward_sample  # noqa: E402

# Global counter of T5 encoder forward calls; hooks that re-encode must call count_enc(k).
n_enc_fwd = 0


def count_enc(k=1):
    global n_enc_fwd
    n_enc_fwd += int(k)


def seed_int(*parts):
    """Stable 63-bit seed from arbitrary parts (process-independent, unlike hash())."""
    h = hashlib.sha256(repr(parts).encode()).digest()
    return int.from_bytes(h[:8], "little") & ((1 << 63) - 1)


class CounterWrapper(nn.Module):
    """Counts batched forward calls of the ELF network."""

    def __init__(self, model):
        super().__init__()
        self.model = model
        self.count = 0

    def forward(self, *args, **kwargs):
        self.count += 1
        return self.model(*args, **kwargs)


def unwrap(model):
    seen = set()
    while id(model) not in seen:
        seen.add(id(model))
        if hasattr(model, "_orig_mod"):
            model = model._orig_mod
        elif isinstance(model, CounterWrapper):
            model = model.model
        else:
            break
    return model


def _device_of(model):
    return next(unwrap(model).parameters()).device


def initial_noise(indices, seed, L, D, noise_scale, device, dtype=torch.float32):
    """z_j = randn(L, D) * noise_scale from Generator(seed*100003 + j)."""
    zs = []
    for j in indices:
        g = torch.Generator(device=device).manual_seed(int(seed) * 100003 + int(j))
        zs.append(torch.randn((L, D), generator=g, device=device, dtype=dtype))
    return torch.stack(zs) * noise_scale


def sample_schedules(indices, seed, n_steps, time_schedule, p_mean, p_std, device, dtype=torch.float32):
    """Per-sample version of utils.sampling_utils.get_sampling_steps -> (B, n_steps+1)."""
    if time_schedule == "uniform":
        t = torch.linspace(0.0, 1.0, n_steps + 1, dtype=dtype)
        return t.unsqueeze(0).repeat(len(indices), 1).to(device)
    if time_schedule not in ("logit_normal", "logit_normal_oversampled"):
        raise ValueError(f"Unsupported time_schedule {time_schedule!r}")
    if time_schedule == "logit_normal_oversampled":
        raise NotImplementedError("logit_normal_oversampled is a training schedule; not used for sampling")
    rows = []
    for j in indices:
        g = torch.Generator(device="cpu").manual_seed(seed_int(seed, int(j), "sched"))
        s = torch.sigmoid(torch.randn((n_steps - 1,), generator=g, dtype=dtype) * p_std + p_mean)
        s = torch.sort(s).values
        rows.append(torch.cat([torch.zeros(1, dtype=dtype), s, torch.ones(1, dtype=dtype)]))
    return torch.stack(rows).to(device)


class PerSampleNoise:
    """SDE noise: eps_j drawn from a per-sample generator seeded by (seed, j, 'sde')."""

    def __init__(self, indices, seed, shape_LD, device, dtype=torch.float32):
        self.gens = [torch.Generator(device=device).manual_seed(seed_int(seed, int(j), "sde")) for j in indices]
        self.shape, self.device, self.dtype = shape_LD, device, dtype

    def __call__(self, step_idx):
        return torch.stack([torch.randn(self.shape, generator=g, device=self.device, dtype=self.dtype)
                            for g in self.gens])


@torch.no_grad()
def rollout(model, z, t_steps, config, method, gamma, sc_cfg, eps_fn=None, hook=None):
    """Euler / SDE rollout. t_steps: (B, n+1) or (n+1,). Returns the final latent z.

    hook(i, t, t_next, t_eff, z, x_pred) -> (z, x_pred) is called after every NON-final step.
    t, t_next, t_eff are (B,) tensors; t_eff = t for ODE and t_back for SDE steps. The returned
    x_pred becomes the self-conditioning input of the next step.
    """
    B, L, D = z.shape
    shared_t = t_steps.dim() == 1
    if shared_t:
        t_1d = t_steps.to(device=z.device, dtype=z.dtype)
        t_steps = t_1d.unsqueeze(0).expand(B, -1)
    else:
        t_steps = t_steps.to(device=z.device, dtype=z.dtype)
    if method not in ("ode", "sde"):
        raise ValueError(f"Invalid sampling method: {method}")
    if method == "sde" and eps_fn is None:
        raise ValueError("SDE sampling needs eps_fn")
    cond_seq = torch.zeros_like(z)
    cond_seq_mask = torch.zeros((B, L), dtype=z.dtype, device=z.device)
    x_pred = torch.zeros_like(z)  # == restore_cond(zeros, zeros, zeros) in the official sampler
    n = t_steps.shape[1]
    use_bf16 = bool(getattr(config, "use_bf16", True)) and z.is_cuda
    with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=use_bf16):
        for i in range(n - 1):
            t, t_next = t_steps[:, i], t_steps[:, i + 1]
            last = i == n - 2
            if shared_t:
                ti, tni = t_1d[i].item(), t_1d[i + 1].item()
                if method == "sde" and not last:
                    h = float(tni - ti)
                    alpha = max(0.0, min(1.0, 1.0 - float(gamma) * h))
                    t_back = alpha * float(ti)
                    eps = eps_fn(i) * config.denoiser_noise_scale
                    z_in = alpha * z + (1.0 - alpha) * eps
                    t_eff = torch.full((B,), t_back, dtype=z.dtype, device=z.device)
                    dt = tni - t_back
                else:
                    z_in = z
                    t_eff = torch.full((B,), float(ti), dtype=z.dtype, device=z.device)
                    dt = tni - ti
            elif method == "sde" and not last:
                t_d, tn_d = t.double(), t_next.double()
                h_d = tn_d - t_d
                alpha_d = (1.0 - float(gamma) * h_d).clamp(0.0, 1.0)
                te_d = alpha_d * t_d
                t_eff = te_d.to(z.dtype)
                eps = eps_fn(i) * config.denoiser_noise_scale
                a = alpha_d.to(z.dtype).view(-1, 1, 1)
                one_m_a = (1.0 - alpha_d).to(z.dtype).view(-1, 1, 1)
                z_in = a * z + one_m_a * eps
                dt = (tn_d - te_d).to(z.dtype).view(-1, 1, 1)
            else:
                t_eff, z_in = t, z
                dt = (t_next - t_eff).view(-1, 1, 1)
            v, x_pred = _forward_sample(
                model=model, z=z_in, t_batch=t_eff, x_pred_prev=x_pred, config=config,
                cfg_scale=1.0, self_cond_cfg_scale=sc_cfg,
                cond_seq=cond_seq, cond_seq_mask=cond_seq_mask,
            )
            z = z_in + dt * v
            if hook is not None and not last:
                z, x_pred = hook(i, t, t_next, t_eff, z, x_pred)
    return z


@torch.no_grad()
def decode_latents(model, z, config, sc_cfg, chunk=32):
    """Official _dlm_decode_batch in chunks (the (B, L, vocab) fp32 logits dominate memory)."""
    out = [_dlm_decode_batch(z=z[s:s + chunk], model=model, t_final_val=1.0, config=config,
                             self_cond_cfg_scale=sc_cfg) for s in range(0, z.shape[0], chunk)]
    return torch.cat(out, 0)


@torch.no_grad()
def generate(model, config, tokenizer, n, seed, steps, method="sde", gamma=1.5, sc_cfg=3.0,
             batch_size=64, hook=None, return_traj=False, time_schedule=None, index_offset=0,
             pad_batches=True, decode_chunk=32, progress=False):
    """Unconditional generation.

    Returns dict(texts, token_ids (n, L) int32, n_elf_fwd, nfe_elf (per sample), n_enc_fwd,
    wall_s, traj (optional)). Same seed => same samples regardless of batch_size.
    """
    global n_enc_fwd
    device = _device_of(model)
    base = unwrap(model)
    L, D = int(config.max_length), int(base.text_encoder_dim)
    ts = time_schedule or getattr(config, "time_schedule", "logit_normal")
    eos_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 1
    pad_id = get_pad_token_id(tokenizer)
    counter = CounterWrapper(model)
    enc0 = n_enc_fwd
    t0 = time.time()
    texts, ids_all, traj = [], [], []
    n_batches = 0
    for start in range(0, n, batch_size):
        idx = list(range(index_offset + start, index_offset + min(start + batch_size, n)))
        real = len(idx)
        if pad_batches and real < batch_size and n > batch_size:
            # Pad to the full batch so torch.compile does not recompile; rows are independent.
            idx = idx + [10**9 + k for k in range(batch_size - real)]
        z = initial_noise(idx, seed, L, D, config.denoiser_noise_scale, device)
        t_steps = sample_schedules(idx, seed, steps, ts, config.denoiser_p_mean, config.denoiser_p_std, device)
        eps_fn = PerSampleNoise(idx, seed, (L, D), device) if method == "sde" else None
        h = hook
        if return_traj and start == 0:
            def h(i, t, tn, te, zz, xp, _inner=hook):
                if _inner is not None:
                    zz, xp = _inner(i, t, tn, te, zz, xp)
                traj.append(dict(i=i, t=t[:4].float().cpu(), x_pred=xp[:4].to(torch.float16).cpu()))
                return zz, xp
        zf = rollout(counter, z, t_steps, config, method, gamma, sc_cfg, eps_fn=eps_fn, hook=h)
        if not torch.isfinite(zf).all():
            raise FloatingPointError(f"non-finite latent in batch starting at {start}")
        pred = decode_latents(counter, zf, config, sc_cfg, chunk=decode_chunk)
        pred = mask_after_eos(pred, eos_token_id=eos_id, pad_token_id=pad_id)[:real]
        ids_all.append(pred.to(torch.int32).cpu().numpy())
        texts.extend(tokenizer.decode(r.tolist(), skip_special_tokens=True) for r in pred.cpu())
        n_batches += 1
        if progress:
            print(f"  generated {start + real}/{n}", end="\r")
    if device.type == "cuda":
        torch.cuda.synchronize()
    out = dict(texts=texts, token_ids=np.concatenate(ids_all, 0) if ids_all else np.zeros((0, L), np.int32),
               n_elf_fwd=counter.count, nfe_elf=counter.count / max(n_batches, 1),
               n_enc_fwd=n_enc_fwd - enc0, wall_s=time.time() - t0)
    if return_traj:
        out["traj"] = traj
    return out


def maybe_compile(model, config, enabled=True, warmup_batch=8):
    """torch.compile(model, dynamic=False) with eager fallback. Returns (model, compile_ok)."""
    if not enabled or not torch.cuda.is_available():
        return model, False
    try:
        from torch import _dynamo
        _dynamo.config.cache_size_limit = max(_dynamo.config.cache_size_limit, 64)
        cm = torch.compile(model, dynamic=False)
        base = unwrap(model)
        L, D = int(config.max_length), int(base.text_encoder_dim)
        z = torch.randn((warmup_batch, L, D), device="cuda") * config.denoiser_noise_scale
        rollout(cm, z, torch.linspace(0, 1, 3, device="cuda"), config, "ode", 0.0, 3.0)
        decode_latents(cm, z, config, 3.0, chunk=warmup_batch)
        torch.cuda.synchronize()
        return cm, True
    except Exception as e:  # pragma: no cover
        print(f"[maybe_compile] torch.compile failed -> eager. {type(e).__name__}: {str(e)[:300]}")
        return model, False


def _is_oom(e):
    return isinstance(e, torch.cuda.OutOfMemoryError) or "out of memory" in str(e).lower()


def find_max_batch(model, config, candidates=(256, 128, 64, 32, 16, 8, 4), method="sde", decode_chunk=32):
    """Largest batch that survives a 3-step rollout + decode at L=config.max_length."""
    import gc
    base = unwrap(model)
    L, D = int(config.max_length), int(base.text_encoder_dim)
    dev = _device_of(model)
    for b in candidates:
        z = zf = None
        try:
            torch.cuda.empty_cache()
            z = torch.randn((b, L, D), device=dev) * config.denoiser_noise_scale
            eps = (lambda i, _b=b: torch.randn((_b, L, D), device=dev)) if method == "sde" else None
            zf = rollout(model, z, torch.linspace(0, 1, 4, device=dev), config, method, 1.0, 3.0, eps_fn=eps)
            decode_latents(model, zf, config, 3.0, chunk=decode_chunk)
            torch.cuda.synchronize()
            del z, zf
            torch.cuda.empty_cache()
            return b
        except Exception as e:
            is_oom = _is_oom(e)
            e.__traceback__ = None
            del z, zf, e
            gc.collect()
            torch.cuda.empty_cache()
            if not is_oom:
                raise
    raise RuntimeError("No batch size fits in memory")

In [ ]:
%%writefile /content/drive/MyDrive/elf_research/code/elfx/metrics.py
"""Metrics that replicate ELF/src/utils/metrics_utils.py exactly, plus extra diagnostics.

* Gen-PPL: CORPUS token-weighted perplexity exp(sum NLL / sum valid tokens), GPT-2 Large in bf16,
  pad = eos, truncation at 1024, first EOS counted, padding masked, batch size 64, empty samples
  dropped before batching -- i.e. exactly what generation.test_generation_uncond does.
  score_gpt2 reuses Metrics._eval_retokenize and Metrics._compute_batch_nlls, so per-sample
  numbers aggregate to the official value.
* Entropy: ELF's per-sample unigram entropy over GPT-2 tokens (nats), averaged over samples.
  NOT decoder softmax entropy.
"""
import math
import os

import numpy as np
import torch

from . import ensure_elf_on_path

ensure_elf_on_path()

PPL_MODEL = os.environ.get("ELFX_PPL_MODEL", "gpt2-large")
# Auto-select GPT-2 Large batch size by VRAM: 16 on <=24GB GPUs (L4/T4), 32 on A100/H100.
_VRAM_GB = (torch.cuda.get_device_properties(0).total_memory / (2 ** 30)) if torch.cuda.is_available() else 0.0
PPL_BATCH = 16 if _VRAM_GB < 35.0 else 32
PPL_MAX_LEN = 1024   # Config.eval_ppl_max_length
_METRICS = {}


def get_ppl_metrics(model_name=None, batch_size=PPL_BATCH, context=PPL_MAX_LEN):
    """An official utils.metrics_utils.Metrics with its eval LM preloaded (bf16 on CUDA)."""
    from transformers import AutoModelForCausalLM
    from utils.metrics_utils import Metrics

    model_name = model_name or PPL_MODEL
    if model_name not in _METRICS:
        m = Metrics(gen_ppl_eval_model_name_or_path=model_name, eval_ppl_batch_size=batch_size,
                    eval_context_size=context)
        dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        lm = AutoModelForCausalLM.from_pretrained(model_name)
        lm = lm.to(device=dev, dtype=torch.bfloat16 if dev.type == "cuda" else torch.float32).eval()
        m._eval_model, m._eval_device = lm, dev
        _METRICS[model_name] = m
    m = _METRICS[model_name]
    m.eval_ppl_batch_size = batch_size
    return m


def free_ppl_model():
    import gc
    for m in _METRICS.values():
        m._eval_model = None
    _METRICS.clear()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _nonempty(texts):
    return [i for i, s in enumerate(texts) if isinstance(s, str) and s.strip()]


def _is_oom(e):
    return isinstance(e, torch.cuda.OutOfMemoryError) or "out of memory" in str(e).lower()


@torch.no_grad()
def score_gpt2(texts, max_length=PPL_MAX_LEN, batch_size=None, model_name=None):
    """Per-sample (nll_sum, token_count, entropy) aligned with `texts`.

    Empty samples get nll=0, count=0, entropy=nan and empty=True (excluded, as officially).
    """
    import gc
    global PPL_BATCH
    batch_size = batch_size or PPL_BATCH
    m = get_ppl_metrics(model_name, batch_size=batch_size, context=max_length)
    n = len(texts)
    keep = _nonempty(texts)
    nll = np.zeros(n, np.float64)
    cnt = np.zeros(n, np.float64)
    ent = np.full(n, np.nan, np.float64)
    if keep:
        sub = [texts[i] for i in keep]
        samples, attn, ctx = m._eval_retokenize(sub, max_length=max_length)
        bs = min(batch_size, samples.shape[0])
        while True:
            try:
                s_nll = np.zeros(len(sub), np.float64)
                s_cnt = np.zeros(len(sub), np.float64)
                for b0 in range(0, samples.shape[0], bs):
                    b1 = min(b0 + bs, samples.shape[0])
                    bsamp, battn = samples[b0:b1], attn[b0:b1]
                    for c0 in range(0, bsamp.shape[1], ctx):
                        c1 = min(c0 + ctx, bsamp.shape[1])
                        ii = torch.from_numpy(bsamp[:, c0:c1]).to(m._eval_device).long()
                        aa = torch.from_numpy(battn[:, c0:c1]).to(m._eval_device).long()
                        nl, vt = m._compute_batch_nlls(ii, aa)
                        nl = nl.detach().cpu().numpy().astype(np.float64)
                        vt = vt.detach().cpu().numpy().astype(np.float64)
                        s_nll[b0:b1] += (nl * vt).sum(-1)
                        s_cnt[b0:b1] += vt.sum(-1)
                break
            except Exception as e:
                is_oom = _is_oom(e)
                e.__traceback__ = None
                del e
                gc.collect()
                torch.cuda.empty_cache()
                if not is_oom or bs == 1:
                    raise
                bs //= 2
                PPL_BATCH = bs
                m.eval_ppl_batch_size = bs
                print(f"[score_gpt2][WARN] OOM -> batch {bs}.")
        # Per-sample unigram entropy, identical to metrics_utils lines 221-228.
        s_ent = np.zeros(len(sub), np.float64)
        for k in range(samples.shape[0]):
            vl = int(attn[k].sum())
            _, counts = np.unique(samples[k, :vl], return_counts=True)
            probs = counts.astype(np.float32) / counts.sum()
            s_ent[k] = float(-np.sum(probs * np.log(probs + 1e-10)))
        nll[keep], cnt[keep], ent[keep] = s_nll, s_cnt, s_ent
    empty = np.ones(n, bool)
    empty[keep] = False
    return dict(nll=nll, count=cnt, entropy=ent, empty=empty)


def corpus_ppl(nll, count):
    nll, count = np.asarray(nll, np.float64), np.asarray(count, np.float64)
    return float(math.exp(nll.sum() / max(count.sum(), 1.0)))


def official_ppl(texts, max_length=PPL_MAX_LEN, model_name=None):
    """Run the official Metrics.record_generative_perplexity (fresh accumulator) for testing."""
    import gc
    from utils.metrics_utils import Metrics

    global PPL_BATCH
    src = get_ppl_metrics(model_name, batch_size=PPL_BATCH)
    bs = int(src.eval_ppl_batch_size or PPL_BATCH)
    keep = [texts[i] for i in _nonempty(texts)]
    while True:
        m = Metrics(gen_ppl_eval_model_name_or_path=src.gen_ppl_eval_model_name_or_path,
                    eval_ppl_batch_size=bs, eval_context_size=max_length)
        m._eval_model, m._eval_device = src._eval_model, src._eval_device
        m.reset()
        try:
            return m.record_generative_perplexity(text_samples=keep, max_length=max_length, retokenize=True)
        except Exception as e:
            is_oom = _is_oom(e)
            e.__traceback__ = None
            del e
            gc.collect()
            torch.cuda.empty_cache()
            if not is_oom or bs <= 1:
                raise
            bs //= 2
            PPL_BATCH = bs
            src.eval_ppl_batch_size = bs
            print(f"[official_ppl][WARN] OOM -> batch {bs}.")


# ---------------------------------------------------------------- text statistics (GPT-2 tokens)
def gpt2_token_ids(texts, max_length=PPL_MAX_LEN, model_name=None):
    tok = get_ppl_metrics(model_name).tokenizer
    return [tok(s, truncation=True, max_length=max_length)["input_ids"] if isinstance(s, str) and s.strip() else []
            for s in texts]


def _ngrams(ids, n):
    return [tuple(ids[i:i + n]) for i in range(len(ids) - n + 1)]


def unigram_entropy(texts=None, ids=None):
    """Mean per-sample unigram entropy (nats) over non-empty samples (ELF definition)."""
    ids = ids if ids is not None else gpt2_token_ids(texts)
    vals = []
    for x in ids:
        if not x:
            continue
        _, c = np.unique(np.asarray(x), return_counts=True)
        p = c.astype(np.float32) / c.sum()
        vals.append(float(-np.sum(p * np.log(p + 1e-10))))
    return float(np.mean(vals)) if vals else float("nan")


def rep_n_per_sample(ids, n=4):
    out = np.full(len(ids), np.nan)
    for k, x in enumerate(ids):
        g = _ngrams(x, n)
        if g:
            out[k] = 1.0 - len(set(g)) / len(g)
    return out


def rep_n(texts=None, n=4, ids=None):
    ids = ids if ids is not None else gpt2_token_ids(texts)
    return float(np.nanmean(rep_n_per_sample(ids, n)))


def distinct_n(texts=None, n=1, ids=None):
    ids = ids if ids is not None else gpt2_token_ids(texts)
    allg = [g for x in ids for g in _ngrams(x, n)]
    return float(len(set(allg)) / len(allg)) if allg else float("nan")


def degeneracy(texts=None, ids=None):
    """Degenerate if rep-4 > 0.5, or unique/len < 0.15, or < 16 GPT-2 tokens."""
    ids = ids if ids is not None else gpt2_token_ids(texts)
    r4 = rep_n_per_sample(ids, 4)
    out = np.zeros(len(ids), bool)
    for k, x in enumerate(ids):
        out[k] = len(x) < 16 or (len(set(x)) / len(x) < 0.15) or (not np.isnan(r4[k]) and r4[k] > 0.5)
    return out


def text_stats(texts):
    ids = gpt2_token_ids(texts)
    r4 = rep_n_per_sample(ids, 4)
    deg = degeneracy(ids=ids)
    return dict(entropy=unigram_entropy(ids=ids), rep4=float(np.nanmean(r4)),
                distinct1=distinct_n(ids=ids, n=1), distinct2=distinct_n(ids=ids, n=2),
                distinct3=distinct_n(ids=ids, n=3), degen=float(deg.mean()),
                mean_gpt2_len=float(np.mean([len(x) for x in ids])),
                per_sample_rep4=r4, per_sample_degen=deg)


# ---------------------------------------------------------------- MAUVE
@torch.no_grad()
def gpt2_features(texts, batch_size=16, max_length=1024, model_name=None):
    """MAUVE features: final-layer (post ln_f) hidden state of the last token, GPT-2 Large."""
    m = get_ppl_metrics(model_name)
    lm, tok = m._eval_model, m.tokenizer
    body = getattr(lm, "transformer", None)
    texts = [s if isinstance(s, str) and s.strip() else tok.eos_token for s in texts]
    feats = []
    for b0 in range(0, len(texts), batch_size):
        enc = tok(texts[b0:b0 + batch_size], return_tensors="pt", truncation=True,
                  max_length=max_length, padding=True)
        ii = enc["input_ids"].to(m._eval_device)
        aa = enc["attention_mask"].to(m._eval_device)
        if body is not None:
            h = body(input_ids=ii, attention_mask=aa).last_hidden_state
        else:
            h = lm(input_ids=ii, attention_mask=aa, output_hidden_states=True).hidden_states[-1]
        last = aa.sum(1) - 1  # GPT-2 tokenizer pads on the right
        feats.append(h[torch.arange(h.shape[0], device=h.device), last].float().cpu().numpy())
    return np.concatenate(feats, 0)


def mauve_from_features(gen_feats, ref_feats, seed=25):
    try:
        import mauve
    except Exception as e:
        print(f"[mauve][WARN] mauve-text not importable ({e}); returning NaN")
        return float("nan")
    try:
        out = mauve.compute_mauve(p_features=ref_feats, q_features=gen_feats, verbose=False, seed=seed)
        return float(out.mauve)
    except Exception as e:
        print(f"[mauve][WARN] compute_mauve failed ({type(e).__name__}: {e}); returning NaN")
        return float("nan")


def mauve_vs_ref(texts, ref_texts=None, ref_feats=None):
    if ref_feats is None:
        ref_feats = gpt2_features(ref_texts)
    return mauve_from_features(gpt2_features(texts), ref_feats)


# ---------------------------------------------------------------- frontier / statistics
def ppl_at_entropy(points, H=5.15):
    """points: iterable of (entropy, ppl) along ONE method's knob. Linear interp of log-PPL."""
    pts = sorted((float(h), math.log(float(p))) for h, p in points if np.isfinite(h) and p > 0)
    for (h1, l1), (h2, l2) in zip(pts, pts[1:]):
        if h1 <= H <= h2:
            w = 0.0 if h2 == h1 else (H - h1) / (h2 - h1)
            return float(math.exp(l1 + w * (l2 - l1)))
    return float("nan")


def bootstrap_ci(per_sample_nll, per_sample_count, B=10000, alpha=0.05, seed=0, chunk=1000):
    """Percentile CI of the corpus PPL, resampling samples with replacement."""
    nll = np.asarray(per_sample_nll, np.float64)
    cnt = np.asarray(per_sample_count, np.float64)
    n = len(nll)
    rng = np.random.default_rng(seed)
    vals = []
    for s in range(0, B, chunk):
        idx = rng.integers(0, n, size=(min(chunk, B - s), n))
        vals.append(np.exp(nll[idx].sum(1) / np.maximum(cnt[idx].sum(1), 1.0)))
    vals = np.concatenate(vals)
    return float(np.quantile(vals, alpha / 2)), float(np.quantile(vals, 1 - alpha / 2))


def paired_bootstrap(nll_a, cnt_a, nll_b, cnt_b, B=10000, seed=0, chunk=1000):
    """Paired resampling over shared sample indices. Returns dict with log-PPL diff CI and p-value
    for H0: PPL_a >= PPL_b (i.e. small p => A has lower PPL)."""
    a_n, a_c = np.asarray(nll_a, np.float64), np.asarray(cnt_a, np.float64)
    b_n, b_c = np.asarray(nll_b, np.float64), np.asarray(cnt_b, np.float64)
    assert len(a_n) == len(b_n), "paired bootstrap needs aligned samples"
    n = len(a_n)
    rng = np.random.default_rng(seed)
    d = []
    for s in range(0, B, chunk):
        idx = rng.integers(0, n, size=(min(chunk, B - s), n))
        la = a_n[idx].sum(1) / np.maximum(a_c[idx].sum(1), 1)
        lb = b_n[idx].sum(1) / np.maximum(b_c[idx].sum(1), 1)
        d.append(la - lb)
    d = np.concatenate(d)
    obs = a_n.sum() / max(a_c.sum(), 1) - b_n.sum() / max(b_c.sum(), 1)
    return dict(log_ppl_diff=float(obs), ci=(float(np.quantile(d, .025)), float(np.quantile(d, .975))),
                p_value=float((d >= 0).mean()))


# ---------------------------------------------------------------- conditional metrics (official)
def bleu_rouge(hypotheses, references):
    from utils.metrics_utils import compute_bleu, compute_rouge
    return dict(bleu=compute_bleu(hypotheses, references), **compute_rouge(hypotheses, references))

In [ ]:
%%writefile /content/drive/MyDrive/elf_research/code/elfx/refs.py
"""OpenWebText reference texts for MAUVE / reference-anchor metrics."""
import json
import os

import numpy as np

from . import REFS, ensure_elf_on_path

ensure_elf_on_path()

OWT_REPO = "embedded-language-flows/openwebtext-t5"


def _decode_t5(rows, tokenizer):
    import torch
    from utils.generation_utils import mask_after_eos

    out = []
    for ids, sl in rows:
        ids = np.asarray(ids, dtype=np.int64)
        if sl is not None:
            ids = ids[: int(sl)]
        t = torch.from_numpy(ids)[None]
        t = mask_after_eos(t, eos_token_id=tokenizer.eos_token_id or 1, pad_token_id=tokenizer.pad_token_id or 0)
        out.append(tokenizer.decode(t[0].tolist(), skip_special_tokens=True))
    return out


def get_owt_refs(n=1000, pool=5000, seed=0, tokenizer=None):
    """n reference docs: rows sampled (seed) from the LAST `pool` rows of the last OWT-T5 shard.

    Cached at REFS/owt_ref_{n}.jsonl. Only one ~0.5 GB shard is downloaded.
    """
    path = os.path.join(REFS, f"owt_ref_{n}.jsonl")
    if os.path.exists(path):
        with open(path) as f:
            return [json.loads(l)["text"] for l in f]
    from transformers import AutoTokenizer

    tokenizer = tokenizer or AutoTokenizer.from_pretrained("t5-small")
    try:
        from datasets import Dataset
        from huggingface_hub import hf_hub_download, list_repo_files

        shards = sorted(f for f in list_repo_files(OWT_REPO, repo_type="dataset") if f.endswith(".arrow"))
        local = hf_hub_download(OWT_REPO, shards[-1], repo_type="dataset")
        ds = Dataset.from_file(local)
        N = len(ds)
        rng = np.random.default_rng(seed)
        rows_idx = (N - pool + rng.choice(pool, size=n, replace=False)).tolist()
        sub = ds.select(rows_idx)
        has_len = "sequence_length" in sub.column_names
        rows = [(r["input_ids"], r["sequence_length"] if has_len else None) for r in sub]
        texts = _decode_t5(rows, tokenizer)
        meta = dict(source=f"{OWT_REPO}:{shards[-1]}", rows=rows_idx, n=n, pool=pool, seed=seed)
    except Exception as e:
        print(f"[refs][WARN] OWT-T5 shard path failed ({type(e).__name__}: {e}); trying Skylion007 fallback")
        from datasets import load_dataset

        ds = load_dataset("Skylion007/openwebtext", split="train", streaming=True)
        buf = []
        for k, r in enumerate(ds):
            if k >= 200_000 + pool:
                break
            if k >= 200_000:
                buf.append(r["text"])
        rng = np.random.default_rng(seed)
        pick = rng.choice(len(buf), size=n, replace=False)
        texts = []
        for i in pick:
            ids = tokenizer(buf[i], truncation=True, max_length=1024)["input_ids"]
            texts.append(tokenizer.decode(ids, skip_special_tokens=True))
        meta = dict(source="Skylion007/openwebtext[stream 200k:+pool] (fallback)", n=n, pool=pool, seed=seed)
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        for t in texts:
            f.write(json.dumps({"text": t}) + "\n")
    os.replace(tmp, path)
    with open(path.replace(".jsonl", "_meta.json"), "w") as f:
        json.dump(meta, f)
    return texts


def get_ref_features(n=1000, model_name=None):
    """GPT-2 Large MAUVE features of the references, cached as .npy on Drive."""
    from .metrics import PPL_MODEL, gpt2_features

    tag = (model_name or PPL_MODEL).replace("/", "_")
    path = os.path.join(REFS, f"owt_ref_{n}_{tag}_feats.npy")
    if os.path.exists(path):
        return np.load(path)
    feats = gpt2_features(get_owt_refs(n), model_name=model_name)
    np.save(path, feats)
    return feats

In [ ]:
%%writefile /content/drive/MyDrive/elf_research/code/elfx/registry.py
"""Run directories, atomic JSON, and the single-source-of-truth registry.csv."""
import csv
import hashlib
import json
import os
import subprocess
import time

from . import CODE, ROOT, RUNS

REGISTRY_FILE = os.path.join(ROOT, "registry.csv")
REGISTRY_COLUMNS = [
    "run_id", "model", "method", "steps", "gamma", "sc_cfg", "hook", "seed", "n",
    "gen_ppl", "ppl_ci_lo", "ppl_ci_hi", "entropy", "mauve", "rep4", "distinct2", "degen",
    "nfe_elf", "nfe_enc", "wall_s", "rouge1", "rouge2", "rougeL", "bleu",
    "git_hash_or_zip_md5", "timestamp",
]


def run_dir(run_id):
    d = os.path.join(RUNS, str(run_id))
    os.makedirs(d, exist_ok=True)
    return d


def is_done(run_id):
    return os.path.exists(os.path.join(RUNS, str(run_id), "summary.json"))


def _jsonable(o):
    try:
        import numpy as np
        if isinstance(o, np.generic):
            return o.item()
        if isinstance(o, np.ndarray):
            return o.tolist()
    except Exception:
        pass
    return str(o)


def save_json_atomic(path, data):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(data, f, indent=2, default=_jsonable)
    os.replace(tmp, path)


def load_json(path, default=None):
    if not os.path.exists(path):
        return default
    with open(path) as f:
        return json.load(f)


def log_jsonl(path, records, mode="w"):
    tmp = path + ".tmp" if mode == "w" else path
    with open(tmp, mode) as f:
        for r in records:
            f.write(json.dumps(r, default=_jsonable) + "\n")
    if mode == "w":
        os.replace(tmp, path)


_CODE_ID = None


def code_id():
    """md5 of CODE/elf_code.zip if present, else git HEAD of the ELF repo, else 'unknown'."""
    global _CODE_ID
    if _CODE_ID is None:
        z = os.path.join(CODE, "elf_code.zip")
        if os.path.exists(z):
            with open(z, "rb") as f:
                _CODE_ID = "zip:" + hashlib.md5(f.read()).hexdigest()[:12]
        else:
            try:
                from . import ELF_REPO
                _CODE_ID = "git:" + subprocess.check_output(
                    ["git", "-C", ELF_REPO, "rev-parse", "--short", "HEAD"], text=True).strip()
            except Exception:
                _CODE_ID = "unknown"
    return _CODE_ID


def append_registry(record):
    """Append one row; rotates an old registry whose header differs."""
    if os.path.exists(REGISTRY_FILE):
        with open(REGISTRY_FILE) as f:
            header = f.readline().strip().split(",")
        if header != REGISTRY_COLUMNS:
            os.replace(REGISTRY_FILE, REGISTRY_FILE.replace(".csv", f"_old_{int(time.time())}.csv"))
    new = not os.path.exists(REGISTRY_FILE)
    row = {k: record.get(k, "") for k in REGISTRY_COLUMNS}
    row["git_hash_or_zip_md5"] = row["git_hash_or_zip_md5"] or code_id()
    row["timestamp"] = row["timestamp"] or time.strftime("%Y-%m-%dT%H:%M:%S")
    for k, v in row.items():
        if isinstance(v, float):
            row[k] = f"{v:.6g}"
    with open(REGISTRY_FILE, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=REGISTRY_COLUMNS)
        if new:
            w.writeheader()
        w.writerow(row)


def read_registry():
    import pandas as pd
    if not os.path.exists(REGISTRY_FILE):
        return pd.DataFrame(columns=REGISTRY_COLUMNS)
    return pd.read_csv(REGISTRY_FILE)

In [ ]:
%%writefile /content/drive/MyDrive/elf_research/code/elfx/sweep.py
"""Idempotent experiment runner: one cfg dict -> samples, metrics, summary.json, registry row.

cfg keys (uncond): run_id, model, method ('ode'|'sde'), steps, gamma, sc_cfg, seed, n,
                   [batch_size], [hook (name in HOOKS)], [mauve (bool, default True)],
                   [time_schedule], [notes]
A sweep is just a list of cfgs; every cfg persists on completion, so a Colab disconnect loses at
most one cfg. Re-running skips finished cfgs (summary.json exists) and reuses saved samples if
generation finished but scoring did not.
"""
import json
import os
import time

import numpy as np
import torch

from . import RUNS
from .registry import append_registry, is_done, load_json, log_jsonl, run_dir, save_json_atomic

# Named hooks for later notebooks (e.g. ECP). Each entry: name -> factory(model, encoder, tokenizer,
# config, **kwargs) -> hook(i, t, t_next, t_eff, z, x_pred) -> (z, x_pred)
HOOKS = {}

_LOADED = {}
_COMPILE = {"enabled": True}
_BATCH_CACHE = os.path.join(RUNS, "_batch_sizes.json")


def set_compile(enabled):
    _COMPILE["enabled"] = bool(enabled)


def unload_all():
    _LOADED.clear()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def get_model(name):
    """Load (and cache) an ELF model; keeps at most one model resident to save VRAM."""
    from .load import load_elf
    from .sample import maybe_compile

    if name not in _LOADED:
        unload_all()
        model, encoder, tokenizer, config = load_elf(name)
        cmodel, ok = maybe_compile(model, config, enabled=_COMPILE["enabled"])
        _LOADED[name] = dict(model=cmodel, eager=model, encoder=encoder, tokenizer=tokenizer,
                             config=config, compile_ok=ok)
        print(f"[get_model] {name}: compile_ok={ok}")
    return _LOADED[name]


def gpu_name():
    return torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"


def batch_size_for(name, method="sde"):
    """Auto batch size (cached per GPU+model in RUNS/_batch_sizes.json)."""
    from .sample import find_max_batch

    cache = load_json(_BATCH_CACHE, {})
    key = f"{gpu_name()}|{name}"
    if key not in cache:
        m = get_model(name)
        cache[key] = find_max_batch(m["eager"], m["config"], method=method)
        save_json_atomic(_BATCH_CACHE, cache)
        print(f"[batch_size_for] {key} -> {cache[key]}")
    return int(cache[key])


def _save_samples(rd, texts, token_ids):
    log_jsonl(os.path.join(rd, "samples.jsonl"), [{"i": i, "text": t} for i, t in enumerate(texts)])
    np.savez_compressed(os.path.join(rd, "tokens.npz"), token_ids=token_ids)


def _load_samples(rd):
    p = os.path.join(rd, "samples.jsonl")
    if not os.path.exists(p) or not os.path.exists(os.path.join(rd, "gen_meta.json")):
        return None
    with open(p) as f:
        texts = [json.loads(l)["text"] for l in f]
    return texts, load_json(os.path.join(rd, "gen_meta.json"))


def score_texts(texts, mauve=True, ref_feats=None, bootstrap_B=10000):
    """All unconditional metrics for a list of texts."""
    from .metrics import bootstrap_ci, corpus_ppl, gpt2_features, mauve_from_features, score_gpt2, text_stats

    t0 = time.time()
    sc = score_gpt2(texts)
    st = text_stats(texts)
    ppl = corpus_ppl(sc["nll"], sc["count"])
    lo, hi = bootstrap_ci(sc["nll"], sc["count"], B=bootstrap_B)
    mv = float("nan")
    if mauve:
        if ref_feats is None:
            from .refs import get_ref_features
            ref_feats = get_ref_features()
        mv = mauve_from_features(gpt2_features(texts), ref_feats)
    out = dict(gen_ppl=ppl, ppl_ci_lo=lo, ppl_ci_hi=hi,
               entropy=float(np.nanmean(sc["entropy"])), entropy_textstats=st["entropy"],
               mauve=mv, rep4=st["rep4"], distinct1=st["distinct1"], distinct2=st["distinct2"],
               distinct3=st["distinct3"], degen=st["degen"], n_empty=int(sc["empty"].sum()),
               mean_gpt2_len=st["mean_gpt2_len"], score_s=time.time() - t0)
    per_sample = dict(nll=sc["nll"], count=sc["count"], entropy=sc["entropy"], empty=sc["empty"],
                      rep4=st["per_sample_rep4"], degen=st["per_sample_degen"])
    return out, per_sample


def run_config(cfg, verbose=True, write_registry=True):
    """Idempotent: generate -> save samples -> score -> summary.json -> registry row."""
    from .sample import generate

    run_id = cfg["run_id"]
    if is_done(run_id):
        s = load_json(os.path.join(RUNS, run_id, "summary.json"))
        if verbose:
            print(f"[skip] {run_id} (done): PPL={s.get('gen_ppl', float('nan')):.2f} H={s.get('entropy', float('nan')):.3f}")
        return s
    rd = run_dir(run_id)
    save_json_atomic(os.path.join(rd, "config.json"), cfg)
    kind = cfg.get("kind", "uncond")

    if kind == "dummy":  # used by the smoke test (registry idempotency)
        summary = dict(cfg, status="done", timestamp=time.strftime("%Y-%m-%dT%H:%M:%S"))
        save_json_atomic(os.path.join(rd, "summary.json"), summary)
        return summary

    if kind == "ref":
        from .refs import get_owt_refs
        texts = get_owt_refs(cfg.get("n", 1000))
        gen_meta = dict(n_elf_fwd=0, nfe_elf=0, n_enc_fwd=0, wall_s=0.0)
        mauve = False
    else:
        cached = _load_samples(rd)
        if cached is not None and len(cached[0]) == cfg["n"]:
            texts, gen_meta = cached
            if verbose:
                print(f"[reuse] {run_id}: found {len(texts)} saved samples; scoring only")
        else:
            m = get_model(cfg["model"])
            bs = cfg.get("batch_size") or batch_size_for(cfg["model"], cfg["method"])
            hook = None
            if cfg.get("hook"):
                hook = HOOKS[cfg["hook"]](m["eager"], m["encoder"], m["tokenizer"], m["config"],
                                          **cfg.get("hook_kwargs", {}))
            g = generate(m["model"], m["config"], m["tokenizer"], n=cfg["n"], seed=cfg["seed"],
                         steps=cfg["steps"], method=cfg["method"], gamma=cfg.get("gamma", 0.0),
                         sc_cfg=cfg.get("sc_cfg", 1.0), batch_size=bs, hook=hook,
                         time_schedule=cfg.get("time_schedule"))
            texts = g["texts"]
            gen_meta = dict(n_elf_fwd=g["n_elf_fwd"], nfe_elf=g["nfe_elf"], n_enc_fwd=g["n_enc_fwd"],
                            wall_s=g["wall_s"], batch_size=bs, compile_ok=m["compile_ok"], gpu=gpu_name())
            _save_samples(rd, texts, g["token_ids"])
            save_json_atomic(os.path.join(rd, "gen_meta.json"), gen_meta)
        mauve = cfg.get("mauve", True)

    metrics, per_sample = score_texts(texts, mauve=mauve)
    np.savez_compressed(os.path.join(rd, "per_sample.npz"), **per_sample)
    summary = dict(cfg)
    summary.update(metrics)
    summary.update(gen_meta)
    summary.update(nfe_enc=gen_meta.get("n_enc_fwd", 0), status="done",
                   timestamp=time.strftime("%Y-%m-%dT%H:%M:%S"))
    save_json_atomic(os.path.join(rd, "summary.json"), summary)
    if write_registry:
        append_registry(summary)
    if verbose:
        print(f"[done] {run_id}: PPL={metrics['gen_ppl']:.2f} [{metrics['ppl_ci_lo']:.2f},{metrics['ppl_ci_hi']:.2f}] "
              f"H={metrics['entropy']:.3f} MAUVE={metrics['mauve']:.3f} rep4={metrics['rep4']:.3f} "
              f"degen={metrics['degen']:.3f} empty={metrics['n_empty']} gen={gen_meta.get('wall_s', 0):.0f}s "
              f"score={metrics['score_s']:.0f}s")
    return summary


def run_sweep(cfgs, stop_on_error=False):
    out = []
    for c in cfgs:
        try:
            out.append(run_config(c))
        except Exception as e:
            print(f"[error] {c.get('run_id')}: {type(e).__name__}: {e}")
            if stop_on_error:
                raise
    return out

In [ ]:
%%writefile /content/drive/MyDrive/elf_research/code/elfx/official.py
"""Runs through the OFFICIAL ELF generation code paths (generation.py), for gates and cross-checks."""
import glob
import json
import os
import time

import torch

from .registry import append_registry, is_done, load_json, run_dir, save_json_atomic
from . import RUNS


def _collect_metrics(off_dir):
    res = []
    for mp in sorted(glob.glob(os.path.join(off_dir, "*", "metrics.jsonl"))):
        with open(mp) as f:
            lines = [json.loads(l) for l in f if l.strip()]
        if lines:
            res.append(dict(name=os.path.basename(os.path.dirname(mp)), **lines[-1]))
    return res


def _seed_all(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def run_cond_official(name, run_id, n=1000, batch_size=64, seed=42, write_registry=True):
    """XSum / WMT14 de-en evaluation via generation.test_generation_cond (eval.py construction).

    Sampling configs come from the model YAML's sampling_configs_path (cond_sampling_configs.yml:
    64-step ODE, CFG=2, SC-CFG=1). Uses the first n validation examples.
    """
    if is_done(run_id):
        s = load_json(os.path.join(RUNS, run_id, "summary.json"))
        print(f"[skip] {run_id}: ROUGE-L={s.get('rougeL')} BLEU={s.get('bleu')}")
        return s
    from generation import test_generation_cond
    from utils.data_utils import load_dataset_split

    from .load import load_elf
    from .sweep import unload_all

    unload_all()
    model, encoder, tokenizer, config, state = load_elf(name, return_state=True)
    del model
    rd = run_dir(run_id)
    off = os.path.join(rd, "official")
    config.output_dir, config.num_samples, config.online_eval = off, n, True
    config.use_compile, config.use_wandb, config.hf_repo_id = False, False, None
    ds = load_dataset_split(config.eval_data_path)
    print(f"[cond] {name}: eval set {config.eval_data_path} ({len(ds)} rows); using first {n}")
    save_json_atomic(os.path.join(rd, "config.json"),
                     dict(run_id=run_id, model=name, kind="cond", n=n, seed=seed, batch_size=batch_size,
                          sampling_configs=[vars(s) for s in config.sampling_configs]))
    gen = torch.Generator(device="cpu").manual_seed(seed)  # eval.py: CPU generator for cond
    _seed_all(seed)
    t0 = time.time()
    for sc in config.sampling_configs:
        test_generation_cond(state=state, encoder=encoder, tokenizer=tokenizer, generator=gen,
                             config=config, sampling_config=sc, dataset=ds,
                             num_samples=n, batch_size=batch_size)
    wall = time.time() - t0
    res = _collect_metrics(off)
    if not res:
        raise RuntimeError(f"No metrics.jsonl produced under {off}")
    sc0, r0 = config.sampling_configs[0], res[0]
    summary = dict(run_id=run_id, model=name, kind="cond", method=sc0.sampling_method,
                   steps=sc0.num_sampling_steps[0], gamma=getattr(sc0, "sde_gamma", 0.0),
                   cfg=sc0.cfgs[0], sc_cfg=sc0.self_cond_cfg_scales[0], seed=seed, n=n,
                   rouge1=r0["rouge1"], rouge2=r0["rouge2"], rougeL=r0["rougeL"], bleu=r0["bleu"],
                   all_results=res, wall_s=wall, status="done", timestamp=time.strftime("%Y-%m-%dT%H:%M:%S"))
    save_json_atomic(os.path.join(rd, "summary.json"), summary)
    if write_registry:
        append_registry(summary)
    print(f"[done] {run_id}: R1={r0['rouge1']:.2f} R2={r0['rouge2']:.2f} RL={r0['rougeL']:.2f} "
          f"BLEU={r0['bleu']:.2f} ({wall:.0f}s)")
    del state, encoder
    torch.cuda.empty_cache()
    return summary


def run_uncond_official(name, run_id, method="sde", steps=32, gamma=1.5, sc_cfg=3.0, n=1000,
                        batch_size=64, seed=0, write_registry=False):
    """Cross-check: generation.test_generation_uncond end-to-end (its own sampler + official PPL).

    NOTE: eval.py passes a CPU torch.Generator; on CUDA, _sde_step then calls
    torch.randn(..., generator=<cpu>, device='cuda'), which raises. We pass a CUDA generator.
    """
    if is_done(run_id):
        return load_json(os.path.join(RUNS, run_id, "summary.json"))
    from configs.config import SamplingConfig
    from generation import test_generation_uncond

    from .load import load_elf
    from .sweep import unload_all

    unload_all()
    model, encoder, tokenizer, config, state = load_elf(name, return_state=True)
    del model, encoder
    rd = run_dir(run_id)
    off = os.path.join(rd, "official")
    config.output_dir, config.num_samples, config.online_eval = off, n, True
    config.use_compile, config.use_wandb, config.hf_repo_id = False, False, None
    sc = SamplingConfig(sampling_method=method, num_sampling_steps=[steps], cfgs=[1], sde_gamma=gamma,
                        self_cond_cfg_scales=[sc_cfg], time_schedule="logit_normal")
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    gen = torch.Generator(device=dev).manual_seed(seed)
    _seed_all(seed)
    t0 = time.time()
    test_generation_uncond(state=state, tokenizer=tokenizer, generator=gen, config=config,
                           sampling_config=sc, num_samples=n, batch_size=batch_size)
    wall = time.time() - t0
    res = _collect_metrics(off)
    if not res:
        raise RuntimeError(f"No metrics.jsonl produced under {off}")
    texts = []
    for p in glob.glob(os.path.join(off, "*", "all_generated_*.jsonl")):
        with open(p) as f:
            texts = [json.loads(l)["generated"] for l in f]
    summary = dict(run_id=run_id, model=name, kind="uncond_official", method=method, steps=steps,
                   gamma=gamma, sc_cfg=sc_cfg, seed=seed, n=n, gen_ppl=res[0]["ppl"],
                   entropy=res[0]["mean_entropy"], wall_s=wall, status="done",
                   timestamp=time.strftime("%Y-%m-%dT%H:%M:%S"))
    if texts:
        from .metrics import corpus_ppl, score_gpt2
        s = score_gpt2(texts)
        summary["elfx_rescored_ppl"] = corpus_ppl(s["nll"], s["count"])
    save_json_atomic(os.path.join(rd, "summary.json"), summary)
    if write_registry:
        append_registry(summary)
    print(f"[done] {run_id}: official PPL={summary['gen_ppl']:.2f} H={summary['entropy']:.3f} "
          f"(elfx rescoring of the same texts: {summary.get('elfx_rescored_ppl', float('nan')):.2f}) {wall:.0f}s")
    del state
    torch.cuda.empty_cache()
    return summary

In [ ]:
# === 1b. IMPORT elfx (re-run after editing any module above) ===
import importlib, logging
logging.basicConfig(level=logging.WARNING, force=True)
for name in ['generation', 'utils.metrics_utils']:
    logging.getLogger(name).setLevel(logging.INFO)   # official progress logs
import elfx
for m in ['elfx', 'elfx.registry', 'elfx.load', 'elfx.sample', 'elfx.metrics', 'elfx.refs', 'elfx.sweep', 'elfx.official']:
    importlib.reload(importlib.import_module(m))
from elfx import load, sample, metrics, refs, registry, sweep, official
sweep.set_compile(USE_COMPILE)
print('elfx', elfx.__version__, '| ROOT =', elfx.ROOT, '| ELF_SRC =', elfx.ELF_SRC)

## 2. Mandatory smoke test (~2–5 min; first-time downloads take longer)
| ID | Check |
|---|---|
| S1 | `load_elf('ELF-B')`: strict keys (0 missing/unexpected), EMA loaded, ~105M params |
| S2 | n=8, 4-step ODE: `token_ids` shape (8, 1024), non-empty texts, finite latents |
| S2b | **Equivalence**: our rollout vs the official `_generate_samples_single_batch` (ODE and SDE), with the same noise and schedule |
| S3 | Determinism: same seed with batch size 4 vs 8 gives identical initial noise and matching tokens; a same-batch rerun is bit-identical |
| S4 | `score_gpt2` aggregate == official `Metrics.record_generative_perplexity` within 0.1% (64 OWT refs) |
| S5 | Registry idempotency (dummy config skipped on rerun) |
| S6 | Throughput: 64 samples, SDE-32, γ=1.5 → projected CCU for the full run |

In [ ]:
# === 2. SMOKE TEST ===
import numpy as np, torch, time, traceback, json, os, shutil
results, notes = {}, {}

def check(name, fn):
    t0 = time.time()
    try:
        msg = fn()
        results[name] = True
        print(f'[PASS] {name} ({time.time()-t0:.0f}s) {msg or ""}')
    except Exception as e:
        results[name] = False
        print(f'[FAIL] {name}: {type(e).__name__}: {e}')
        traceback.print_exc(limit=3)

if RUN_SMOKE:
    st = {}

    def s1():
        m = sweep.get_model('ELF-B')
        info = m['eager']._elfx_info
        assert info['missing_keys'] == [] and info['unexpected_keys'] == []
        assert info['ema_loaded'], 'EMA weights not loaded'
        assert abs(info['n_params'] - 105e6) / 105e6 < 0.03, info['n_params']
        st.update(m)
        return f"{info['n_params']/1e6:.1f}M params, step {info['step']}, compile_ok={m['compile_ok']}"
    check('S1 load', s1)

    def s2():
        r = sample.generate(st['model'], st['config'], st['tokenizer'], n=8, seed=0, steps=4,
                            method='ode', gamma=0.0, sc_cfg=3.0, batch_size=8)
        assert r['token_ids'].shape == (8, st['config'].max_length), r['token_ids'].shape
        n_empty = sum(1 for t in r['texts'] if not t.strip())
        assert n_empty <= 1, f'{n_empty}/8 empty texts'
        st['r8'] = r
        return f"nfe/sample={r['nfe_elf']:.0f} | sample[0]: {r['texts'][0][:150]!r}"
    check('S2 generate', s2)

    def s2b():
        from utils.generation_utils import _generate_samples_single_batch
        from utils.sampling_utils import get_sampling_steps
        from configs.config import SamplingConfig
        model, cfg = st['eager'], st['config']
        L, D = cfg.max_length, model.text_encoder_dim
        z = sample.initial_noise(range(4), 7, L, D, cfg.denoiser_noise_scale, 'cuda')
        torch.manual_seed(11)
        ts = get_sampling_steps(n_steps=8, time_schedule='logit_normal', P_mean=cfg.denoiser_p_mean,
                                P_std=cfg.denoiser_p_std, device='cuda')
        out = []
        for method, gamma in [('ode', 0.0), ('sde', 1.5)]:
            sc = SamplingConfig(sampling_method=method, num_sampling_steps=[8], cfgs=[1], sde_gamma=gamma,
                                self_cond_cfg_scales=[3], time_schedule='logit_normal')
            g1 = torch.Generator(device='cuda').manual_seed(123)
            off = _generate_samples_single_batch(model=model, generator=g1, z=z.clone(), t_steps=ts,
                    cond_seq=None, cond_seq_mask=None, config=cfg, sampling_config=sc,
                    cfg_scale=1, self_cond_cfg_scale=3)
            g2 = torch.Generator(device='cuda').manual_seed(123)
            eps = (lambda i: torch.randn(z.shape, generator=g2, device='cuda')) if method == 'sde' else None
            mine = sample.rollout(model, z.clone(), ts, cfg, method, gamma, 3.0, eps_fn=eps)
            rel = ((off - mine).norm() / off.norm()).item()
            a = sample.decode_latents(model, off, cfg, 3.0); b = sample.decode_latents(model, mine, cfg, 3.0)
            agree = (a == b).float().mean().item()
            assert rel < 1e-3 and agree > 0.995, f'{method}: rel_err={rel:.2e} token_agree={agree:.4f}'
            out.append(f'{method}: rel_err={rel:.1e} agree={agree:.4f}')
        return ' | '.join(out)
    check('S2b equivalence vs official sampler', s2b)

    def s3():
        cfg, L, D = st['config'], st['config'].max_length, st['eager'].text_encoder_dim
        z_a = sample.initial_noise(range(8), 0, L, D, cfg.denoiser_noise_scale, 'cuda')
        z_b = torch.cat([sample.initial_noise(range(0, 4), 0, L, D, cfg.denoiser_noise_scale, 'cuda'),
                         sample.initial_noise(range(4, 8), 0, L, D, cfg.denoiser_noise_scale, 'cuda')])
        assert torch.equal(z_a, z_b), 'initial noise depends on batch layout'
        ts_a = sample.sample_schedules(range(8), 0, 4, 'logit_normal', cfg.denoiser_p_mean, cfg.denoiser_p_std, 'cuda')
        ts_b = torch.cat([sample.sample_schedules(range(0, 4), 0, 4, 'logit_normal', cfg.denoiser_p_mean, cfg.denoiser_p_std, 'cuda'),
                          sample.sample_schedules(range(4, 8), 0, 4, 'logit_normal', cfg.denoiser_p_mean, cfg.denoiser_p_std, 'cuda')])
        assert torch.equal(ts_a, ts_b), 'schedule depends on batch layout'
        eps_a = sample.PerSampleNoise(range(8), 0, (L, D), 'cuda')(0)
        eps_b = torch.cat([sample.PerSampleNoise(range(0, 4), 0, (L, D), 'cuda')(0),
                           sample.PerSampleNoise(range(4, 8), 0, (L, D), 'cuda')(0)])
        assert torch.equal(eps_a, eps_b), 'SDE noise depends on batch layout'
        kw = dict(n=8, seed=0, steps=4, method='sde', gamma=1.5, sc_cfg=3.0)
        r4 = sample.generate(st['model'], cfg, st['tokenizer'], batch_size=4, **kw)
        r8 = sample.generate(st['model'], cfg, st['tokenizer'], batch_size=8, **kw)
        r8b = sample.generate(st['model'], cfg, st['tokenizer'], batch_size=8, **kw)
        assert np.array_equal(r8['token_ids'], r8b['token_ids']), 'same-batch rerun not deterministic'
        agree = (r4['token_ids'] == r8['token_ids']).mean()
        assert agree >= 0.95, f'bs4 vs bs8 token agreement {agree:.5f}'
        return f'same-batch exact=True | RNG batch-invariant=True | bs4 vs bs8 bf16 agreement={agree:.5f}'
    check('S3 determinism', s3)

    def s4():
        ref = refs.get_owt_refs(1000)[:64]
        s = metrics.score_gpt2(ref)
        mine = metrics.corpus_ppl(s['nll'], s['count'])
        off = metrics.official_ppl(ref)
        metrics.free_ppl_model()
        rel = abs(mine - off['ppl']) / off['ppl']
        dH = abs(np.nanmean(s['entropy']) - off['mean_entropy'])
        assert rel < 1e-3, f'PPL mismatch: elfx {mine:.4f} vs official {off["ppl"]:.4f}'
        assert dH < 1e-6, f'entropy mismatch {dH}'
        return f'PPL elfx={mine:.4f} official={off["ppl"]:.4f} | H={off["mean_entropy"]:.4f}'
    check('S4 metrics == official', s4)

    def s5():
        rid = '_smoke_dummy'
        shutil.rmtree(registry.run_dir(rid), ignore_errors=True)
        a = sweep.run_config(dict(run_id=rid, kind='dummy', x=1), write_registry=False)
        b = sweep.run_config(dict(run_id=rid, kind='dummy', x=2), write_registry=False)
        assert a['x'] == 1 and b['x'] == 1, 'rerun was not skipped'
        shutil.rmtree(registry.run_dir(rid), ignore_errors=True)
        return 'second call skipped'
    check('S5 registry idempotency', s5)

    def s6():
        cfg = st['config']
        bs = sweep.batch_size_for('ELF-B', 'sde')
        sample.generate(st['model'], cfg, st['tokenizer'], n=min(bs, 64), seed=99, steps=4,
                        method='sde', gamma=1.5, sc_cfg=3.0, batch_size=min(bs, 64))   # warm-up
        r = sample.generate(st['model'], cfg, st['tokenizer'], n=64, seed=123, steps=32,
                            method='sde', gamma=1.5, sc_cfg=3.0, batch_size=min(bs, 64))
        t0 = time.time(); metrics.score_gpt2(r['texts']); t_score = time.time() - t0
        t_feat0 = time.time(); metrics.gpt2_features(r['texts']); t_feat = time.time() - t_feat0
        gpu = torch.cuda.get_device_name(0)
        rate = next((v for k, v in CCU_PER_HOUR.items() if k in gpu), 12.0)
        gen1k = r['wall_s'] / 64 * 1000          # ELF-B, 32 steps
        score1k = (t_score + t_feat) / 64 * 1000
        # Rough scaling: cost ~ params x NFE. ELF-M ~3.3x, ELF-L ~6.2x ELF-B.
        plan = [('R1 ELF-B ODE-32', 3, 1.0), ('R1 ELF-B SDE-32', 3, 1.0), ('R1 ELF-B SDE-64', 3, 65/33),
                ('R1b official cross-check', 1, 1.1), ('R2 ELF-M SDE-64', 1, 3.3*65/33), ('R2 ELF-L SDE-64', 1, 6.2*65/33)]
        rows, tot_h = [], 0.0
        for name, k, mult in plan:
            h = k * (gen1k * mult + score1k) / 3600
            tot_h += h; rows.append((name, k, h, h * rate))
        h_cond = 0.25; tot_h += h_cond; rows.append(('R3 XSum+WMT (official)', 2, h_cond, h_cond * rate))
        print(f'\n  GPU={gpu} rate={rate} CCU/h | batch={bs} | ELF-B SDE-32: {gen1k:.0f}s/1000 samples | '
              f'scoring+MAUVE feats: {score1k:.0f}s/1000')
        print(f'  {"block":28s} {"cfgs":>4s} {"hours":>6s} {"CCU":>6s}')
        for name, k, h, c in rows:
            print(f'  {name:28s} {k:4d} {h:6.2f} {c:6.1f}')
        print(f'  {"TOTAL (excl. downloads)":28s} {"":4s} {tot_h:6.2f} {tot_h*rate:6.1f}')
        registry.save_json_atomic(f'{RUNS}/_throughput.json', dict(
            gpu=gpu, ccu_per_hour=rate, batch_size=bs, compile_ok=st['compile_ok'],
            elfb_sde32_s_per_1000=gen1k, score_s_per_1000=score1k,
            plan=[dict(block=n, cfgs=k, hours=h, ccu=c) for n, k, h, c in rows],
            total_hours=tot_h, total_ccu=tot_h * rate))
        return f'projected full run ~{tot_h*rate:.0f} CCU'
    check('S6 throughput', s6)

    print('\n' + '=' * 70)
    failed = [k for k, v in results.items() if not v]
    registry.save_json_atomic(f'{RUNS}/_smoke.json', dict(results=results, env=env))
    if failed:
        raise RuntimeError(f'SMOKE TEST FAILED: {failed}. Fix before running anything else.')
    print('SMOKE TEST PASSED - safe to run the full baseline section.')

## 3. Full run (~40 CCU on A100)
- R0: reference anchor (1000 OWT docs)
- R1: ELF-B ODE-32, SDE-32 γ=1.5, and SDE-64 γ=1.0, each at SC-CFG=3, × 3 seeds
- R1b: official end-to-end cross-check
- R2: ELF-M / ELF-L
- R3: XSum / WMT

Each cell can be re-run safely; finished configs are skipped.

In [ ]:
# === R0: reference anchor + cached MAUVE reference features ===
_smoke = registry.load_json(f'{RUNS}/_smoke.json', {})
assert _smoke.get('results') and all(_smoke['results'].values()), 'Smoke test has not passed yet (runs/_smoke.json).'
if RUN_FULL:
    sweep.run_config(dict(run_id='R0_owt_ref', kind='ref', model='OWT-ref', method='-', n=N_SAMPLES, seed=0))
    _ = refs.get_ref_features(N_SAMPLES)   # cached on Drive for MAUVE

In [ ]:
# === R1: ELF-B unconditional baselines (paper SDE-32 g1.5: 24.1 PPL / 5.15 H) ===
R1 = [('ode', 32, 0.0, 3.0), ('sde', 32, 1.5, 3.0), ('sde', 64, 1.0, 3.0)]
if RUN_FULL:
    cfgs = [dict(run_id=f'R1_ELF-B_{m}{s}_g{g}_sc{c:g}_s{seed}', model='ELF-B', method=m, steps=s, gamma=g,
                 sc_cfg=c, seed=seed, n=N_SAMPLES, mauve=True)
            for (m, s, g, c) in R1 for seed in SEEDS]
    sweep.run_sweep(cfgs)

In [ ]:
# === R1b: official end-to-end cross-check (generation.test_generation_uncond, its own sampler + Metrics) ===
if RUN_FULL and RUN_OFFICIAL_CROSSCHECK:
    bs = sweep.batch_size_for('ELF-B', 'sde')
    xc = official.run_uncond_official('ELF-B', 'R1b_ELF-B_official_sde32_g1.5_sc3_s0', method='sde', steps=32,
                                      gamma=1.5, sc_cfg=3.0, n=N_SAMPLES, batch_size=min(bs, 128), seed=0)
    ours = registry.load_json(f'{RUNS}/R1_ELF-B_sde32_g1.5_sc3_s0/summary.json')
    if ours:
        print(f"official PPL {xc['gen_ppl']:.2f} vs elfx PPL {ours['gen_ppl']:.2f} "
              f"[{ours['ppl_ci_lo']:.2f}, {ours['ppl_ci_hi']:.2f}] | H {xc['entropy']:.3f} vs {ours['entropy']:.3f}")
        if not (ours['ppl_ci_lo'] * 0.95 <= xc['gen_ppl'] <= ours['ppl_ci_hi'] * 1.05):
            print('[WARN] official and elfx PPL disagree beyond CI+5%: diff the samplers before going on.')

In [ ]:
# === R2: scale (ELF-M paper 21.7 / 5.18; ELF-L paper 23.3 / 5.28) ===
if RUN_FULL:
    cfgs = []
    if RUN_ELF_M:
        cfgs.append(dict(run_id='R2_ELF-M_sde64_g1.0_sc3_s0', model='ELF-M', method='sde', steps=64, gamma=1.0,
                         sc_cfg=3.0, seed=0, n=N_SAMPLES, mauve=True))
    if RUN_ELF_L:
        cfgs.append(dict(run_id='R2_ELF-L_sde64_g1.0_sc3_s0', model='ELF-L', method='sde', steps=64, gamma=1.0,
                         sc_cfg=3.0, seed=0, n=N_SAMPLES, mauve=True))
    sweep.run_sweep(cfgs)
    sweep.unload_all()

In [ ]:
# === R3: conditional via the OFFICIAL path (64-step ODE, CFG=2, SC-CFG=1; first 1000 val examples) ===
if RUN_FULL and RUN_COND:
    official.run_cond_official('ELF-B-xsum', 'R3_ELF-B-xsum_ode64_cfg2', n=1000, batch_size=64, seed=42)
    official.run_cond_official('ELF-B-de-en', 'R3_ELF-B-de-en_ode64_cfg2', n=1000, batch_size=128, seed=42)

## 4. Gates and baseline table

In [ ]:
# === GATES -> runs/_gate.json ===
import numpy as np, glob
def S(rid): return registry.load_json(f'{RUNS}/{rid}/summary.json')
sde = [S(f'R1_ELF-B_sde32_g1.5_sc3_s{s}') for s in SEEDS]
sde = [x for x in sde if x]
gate = {}
if sde:
    ppl = float(np.mean([x['gen_ppl'] for x in sde])); H = float(np.mean([x['entropy'] for x in sde]))
    gate['G-uncond'] = dict(ppl=ppl, entropy=H, n_seeds=len(sde),
                            pass_=bool(22.5 <= ppl <= 26.0 and 5.08 <= H <= 5.22 and len(sde) == len(SEEDS)))
xs, wm = S('R3_ELF-B-xsum_ode64_cfg2'), S('R3_ELF-B-de-en_ode64_cfg2')
if xs and wm:
    gate['G-cond'] = dict(xsum_rougeL=xs['rougeL'], wmt_bleu=wm['bleu'],
                          pass_=bool(xs['rougeL'] >= 26.5 and wm['bleu'] >= 24.5))
gate['ALL_PASS'] = bool(gate) and all(v['pass_'] for v in gate.values() if isinstance(v, dict))
registry.save_json_atomic(f'{RUNS}/_gate.json', gate)
print(json.dumps(gate, indent=2))
if not gate.get('ALL_PASS'):
    print('\nGATE NOT PASSED: do NOT start Prompts 1A/2A. Compare R1 vs R1b and diff elfx/sample.py vs generation.py.')

In [ ]:
# === BASELINES TABLE -> runs/_baselines_table.md ===
import pandas as pd
rows = []
for p in sorted(glob.glob(f'{RUNS}/R*/summary.json')):
    s = json.load(open(p)); rows.append(s)
df = pd.DataFrame(rows)
unc = df[df.get('kind', pd.Series(['uncond'] * len(df))).fillna('uncond').isin(['uncond', 'ref'])].copy()
lines = ['| Config | Seeds | Gen-PPL (mean) | per-seed 95% CI | Entropy | MAUVE | rep-4 | degen | NFE |',
         '|---|---|---|---|---|---|---|---|---|']
if len(unc):
    unc['cfg_key'] = unc['run_id'].str.replace(r'_s\d+$', '', regex=True)
    for key, g in unc.groupby('cfg_key', sort=False):
        ci = '; '.join(f"[{a:.1f},{b:.1f}]" for a, b in zip(g['ppl_ci_lo'], g['ppl_ci_hi']))
        sd = f" ± {g['gen_ppl'].std():.2f}" if len(g) > 1 else ''
        lines.append(f"| {key} | {len(g)} | {g['gen_ppl'].mean():.2f}{sd} | {ci} | {g['entropy'].mean():.3f} | "
                     f"{g['mauve'].mean():.3f} | {g['rep4'].mean():.3f} | {g['degen'].mean():.3f} | "
                     f"{g.get('nfe_elf', pd.Series([np.nan])).mean():.0f} |")
cond = df[df.get('kind', pd.Series([''] * len(df))) == 'cond']
if len(cond):
    lines += ['', '| Conditional | ROUGE-1 | ROUGE-2 | ROUGE-L | BLEU |', '|---|---|---|---|---|']
    for _, r in cond.iterrows():
        lines.append(f"| {r['run_id']} | {r['rouge1']:.2f} | {r['rouge2']:.2f} | {r['rougeL']:.2f} | {r['bleu']:.2f} |")
lines += ['', 'Paper reference: ELF-B SDE-32 γ1.5 SC-CFG3 = 24.1 PPL / 5.15 H; ELF-M SDE-64 γ1.0 = 21.7 / 5.18; ELF-L SDE-64 γ1.0 = 23.3 / 5.28.']
md_table = '\n'.join(lines)
open(f'{RUNS}/_baselines_table.md', 'w').write(md_table)
from IPython.display import Markdown, display
display(Markdown(md_table))

## Outputs (all under `MyDrive/elf_research/`)
- `code/elfx/*`: importable harness for Prompts 1A, 1B, 2A, 2B and 4
- `registry.csv`: one row per run; this is the single source of truth
- `runs/<run_id>/{config.json, summary.json, samples.jsonl, tokens.npz, per_sample.npz}`. Metrics can be recomputed from these without regenerating.
- `runs/_gate.json`, `runs/_baselines_table.md`, `runs/_throughput.json`, `runs/_smoke.json`, `runs/_env.json`

**Troubleshooting**
- *Drive not mounted / path errors*: `ROOT` in cell 0a and the `%%writefile` paths must agree. They default to `/content/drive/MyDrive/elf_research`.
- *torch.compile errors*: set `USE_COMPILE = False`. The results are identical; compile only affects speed.
- *OOM*: delete `runs/_batch_sizes.json` to re-probe, or pass `batch_size=` in a cfg.
- *A run looks wrong*: delete its `runs/<run_id>/` folder and re-run the cell.